# 02 – Load and validate

Loads the extracted games into DuckDB and checks the data before analysis.
Each check records what was tested, what was found, and the cleaning decision.

**Requires:** `data/processed/games.csv` (created by `src/extract_headers.py`).
**Run:** top to bottom. Paths are relative to the project root.

## Setup and load

In [1]:
import os
import duckdb

# Work from the project root so every path matches the script and the SQL files.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print(os.getcwd())

database_connection = duckdb.connect("data/lichess.duckdb")

/Users/alec/lichess/lichess-ratings


## 01_load.sql

In [2]:
database_connection.execute("DROP TABLE IF EXISTS games_raw")

with open("sql/01_load.sql") as sql_file:
    database_connection.execute(sql_file.read())

database_connection.sql("SELECT COUNT(*) AS total_rows FROM games_raw").show()

┌────────────┐
│ total_rows │
│   int64    │
├────────────┤
│    4770357 │
└────────────┘



In [3]:
database_connection.sql("""
SELECT * FROM games_raw LIMIT 5;
""").show()

┌─────────────────────────────────────────────────────────────────┬──────────────────────────────┬──────────────────┬──────────────┬─────────┬────────────┬──────────┬──────────┬──────────┬─────────────────┬─────────────────┬─────────┬─────────────────────────────────────┬─────────────┬─────────────┬────────────┬────────────┐
│                              Event                              │             Site             │      White       │    Black     │ Result  │  UTCDate   │ UTCTime  │ WhiteElo │ BlackElo │ WhiteRatingDiff │ BlackRatingDiff │   ECO   │               Opening               │ TimeControl │ Termination │ WhiteTitle │ BlackTitle │
│                             varchar                             │           varchar            │     varchar      │   varchar    │ varchar │  varchar   │ varchar  │ varchar  │ varchar  │     varchar     │     varchar     │ varchar │               varchar               │   varchar   │   varchar   │  varchar   │  varchar   │
├──────────────────

In [4]:
database_connection.sql("""
DESCRIBE games_raw;
""").show()

┌─────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│   column_name   │ column_type │  null   │   key   │ default │  extra  │
│     varchar     │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ Event           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Site            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ White           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Black           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Result          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ UTCDate         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ UTCTime         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ WhiteElo        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ BlackElo        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ WhiteRatingDiff │ VARCHAR     │ YES 

## Check 1: Duplicate games

In [5]:
database_connection.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(Site) AS total_sites,
    COUNT(DISTINCT Site) AS distinct_sites
FROM games_raw;
""").show()

┌────────────┬─────────────┬────────────────┐
│ total_rows │ total_sites │ distinct_sites │
│   int64    │    int64    │     int64      │
├────────────┼─────────────┼────────────────┤
│    4770357 │     4770357 │        4770357 │
└────────────┴─────────────┴────────────────┘



Site is each game’s URL and should be unique. All three counts are 4,770,357: every game has an ID and none repeat. 0 games removed.

## Check 2: Result values

In [6]:
database_connection.sql("""
SELECT
    Result,
    COUNT(*) AS games,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS percent_of_games
FROM games_raw
GROUP BY Result
ORDER BY games DESC;
""").show()

┌─────────┬─────────┬──────────────────┐
│ Result  │  games  │ percent_of_games │
│ varchar │  int64  │      double      │
├─────────┼─────────┼──────────────────┤
│ 1-0     │ 2372888 │            49.74 │
│ 0-1     │ 2223679 │            46.61 │
│ 1/2-1/2 │  173790 │             3.64 │
└─────────┴─────────┴──────────────────┘



Exactly three values, all scorable: White wins 49.74%, Black wins 46.61%, draws 3.64%. No * (unfinished) games. 0 removed. White’s unadjusted edge is examined in the analysis.

## Check 3: Termination types

In [7]:
database_connection.sql("""
SELECT
    Termination,
    COUNT(*) AS games,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 4) AS percent_of_games
FROM games_raw
GROUP BY Termination
ORDER BY games DESC;
""").show()

┌──────────────────┬─────────┬──────────────────┐
│   Termination    │  games  │ percent_of_games │
│     varchar      │  int64  │      double      │
├──────────────────┼─────────┼──────────────────┤
│ Normal           │ 3148373 │          65.9987 │
│ Time forfeit     │ 1601611 │          33.5742 │
│ Abandoned        │   20220 │           0.4239 │
│ Rules infraction │     153 │           0.0032 │
└──────────────────┴─────────┴──────────────────┘



Checked percent results and total results match. Found Rules infraction type for Termination and will exclude Abandoned and Rules infraction (20,373 games) as the outcome of these games wasn’t decided by play.  *May not sum to 100 due to rounding*

## Check 4: Ratings

### 4a: Unusable ratings

In [8]:
database_connection.sql("""

SELECT 
    SUM(CASE WHEN TRY_CAST(WhiteElo AS INTEGER) IS NULL THEN 1 ELSE 0 END) AS white_elo_try_cast_null,
    SUM(CASE WHEN WhiteElo IS NULL THEN 1 ELSE 0 END) AS white_elo_null,
    SUM(CASE WHEN WhiteElo = '?' THEN 1 ELSE 0 END) AS white_elo_question_mark,
    SUM(CASE WHEN TRY_CAST(BlackElo AS INTEGER) IS NULL THEN 1 ELSE 0 END) AS black_elo_try_cast_null,
    SUM(CASE WHEN BlackElo IS NULL THEN 1 ELSE 0 END) AS black_elo_null,
    SUM(CASE WHEN BlackElo = '?' THEN 1 ELSE 0 END) AS black_elo_question_mark,
    SUM(CASE WHEN BlackElo = '?' OR WhiteElo = '?' THEN 1 ELSE 0 END) AS games_with_unusable_rating
FROM games_raw

""").show()


┌─────────────────────────┬────────────────┬─────────────────────────┬─────────────────────────┬────────────────┬─────────────────────────┬────────────────────────────┐
│ white_elo_try_cast_null │ white_elo_null │ white_elo_question_mark │ black_elo_try_cast_null │ black_elo_null │ black_elo_question_mark │ games_with_unusable_rating │
│         int128          │     int128     │         int128          │         int128          │     int128     │         int128          │           int128           │
├─────────────────────────┼────────────────┼─────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┼────────────────────────────┤
│                      11 │              0 │                      11 │                      26 │              0 │                      26 │                         37 │
└─────────────────────────┴────────────────┴─────────────────────────┴─────────────────────────┴────────────────┴─────────────────────────┴────────────────

Converted `WhiteElo` and `BlackElo` to integers with `TRY_CAST`.
11 White and 26 Black ratings fail to convert, and every one of them is `'?'` (no `NULL`s, no other
non-numeric values). 37 games have at least one unusable rating; no game has both.
**Decision:** exclude these 37 games (0.0008% of the month), because the Elo formula needs both
players' ratings.

### 4b: Rating range

In [9]:
database_connection.sql("""

SELECT
    MIN(TRY_CAST(WhiteElo AS INTEGER)) AS min_white_elo,
    MAX(TRY_CAST(WhiteElo AS INTEGER)) AS max_white_elo,
    MIN(TRY_CAST(BlackElo AS INTEGER)) AS min_black_elo,
    MAX(TRY_CAST(BlackElo AS INTEGER)) AS max_black_elo
FROM games_raw

""").show()

┌───────────────┬───────────────┬───────────────┬───────────────┐
│ min_white_elo │ max_white_elo │ min_black_elo │ max_black_elo │
│     int32     │     int32     │     int32     │     int32     │
├───────────────┼───────────────┼───────────────┼───────────────┤
│           739 │          2998 │           740 │          2995 │
└───────────────┴───────────────┴───────────────┴───────────────┘



Find the lowest and highest rating of the month to check if they're plausible. Absurd value (ex 0 or 9000) could distort chart and analysis.
**Finding:** No ratings are an absurd amount.  min_white_elo=739, min_black_elo=740, max_white_elo=2998, max_black_elo=2995. 0 removed

### 4c: The 1500 starting rating

In [10]:
database_connection.sql("""

SELECT
    TRY_CAST(WhiteElo AS INTEGER) AS white_rating,
    COUNT(*) AS games
FROM games_raw
WHERE white_rating BETWEEN 1490 AND 1510
GROUP BY white_rating
ORDER BY white_rating


""").show()

database_connection.sql("""

SELECT
    SUM(CASE WHEN TRY_CAST(WhiteElo AS INTEGER) = 1500 OR TRY_CAST(BlackElo AS INTEGER) = 1500 THEN 1 ELSE 0 END) AS games_with_either_player_1500_elo
FROM games_raw

""").show()

┌──────────────┬───────┐
│ white_rating │ games │
│    int32     │ int64 │
├──────────────┼───────┤
│         1490 │  4741 │
│         1491 │  4894 │
│         1492 │  4904 │
│         1493 │  4716 │
│         1494 │  4936 │
│         1495 │  4910 │
│         1496 │  5066 │
│         1497 │  4911 │
│         1498 │  5018 │
│         1499 │  4936 │
│         1500 │ 40540 │
│         1501 │  5021 │
│         1502 │  5068 │
│         1503 │  5034 │
│         1504 │  5074 │
│         1505 │  5086 │
│         1506 │  5123 │
│         1507 │  5101 │
│         1508 │  5080 │
│         1509 │  5215 │
│         1510 │  5106 │
└──────────────┴───────┘
  21 rows    2 columns

┌───────────────────────────────────┐
│ games_with_either_player_1500_elo │
│              int128               │
├───────────────────────────────────┤
│                             70899 │
└───────────────────────────────────┘



Test whether 1500, Lichess's starting rating, is unusually common. Accounts started are given 1500 as an initial placeholder before their first game.
**Finding:** The 1500 rating has 40540 games, about 8 times more than all the other Elo values 1490-1510 which are around 5000 games. 70,899 games (1.5%) have at least one player at 1500.
**Decision:** Keep the 1500 games, but flag them. Add a true/false column in clean table then run the main calibration both ways, with and without flagged games. Decided to go this route instead of deleting all 1500 rated games so its effect can be measured with a sensitivity analysis. Might be an important takeaway from this project. 
**Limitation:** The flag only catches each player’s first game in a time control, so the true effect of unreliable ratings is likely larger. Lichess keeps a separate rating per time control, so 1500 also marks experienced players new to a speed.

## Check 5: Time controls

### 5a: Time control values

In [11]:
database_connection.sql("""

SELECT 
    COUNT(DISTINCT TimeControl) AS distinct_time_control
FROM games_raw

""").show()

database_connection.sql("""

SELECT
    TimeControl,
    COUNT(*) AS games
FROM games_raw
GROUP BY TimeControl
ORDER BY games DESC
LIMIT 15

""").show()

┌───────────────────────┐
│ distinct_time_control │
│         int64         │
├───────────────────────┤
│                   856 │
└───────────────────────┘

┌─────────────┬────────┐
│ TimeControl │ games  │
│   varchar   │ int64  │
├─────────────┼────────┤
│ 60+0        │ 882796 │
│ 300+0       │ 661672 │
│ 180+0       │ 613479 │
│ 600+0       │ 319257 │
│ 30+0        │ 286328 │
│ 120+0       │ 166389 │
│ 180+2       │ 125455 │
│ 300+8       │ 105060 │
│ 900+0       │ 104229 │
│ 480+0       │  75457 │
│ 300+5       │  70923 │
│ 0+1         │  62159 │
│ 120+1       │  58020 │
│ 240+0       │  56558 │
│ 420+0       │  54965 │
└─────────────┴────────┘
  15 rows    2 columns



### 5b: Parsing base and increment

In [12]:
database_connection.sql("""

SELECT
    SUM(CASE WHEN TRY_CAST(split_part(TimeControl, '+', 1) AS INTEGER) IS NULL THEN 1 ELSE 0 END) AS base_is_null,
    SUM(CASE WHEN TRY_CAST(split_part(TimeControl, '+', 2) AS INTEGER) IS NULL THEN 1 ELSE 0 END) AS increment_is_null,
    SUM(CASE WHEN TimeControl = '-' THEN 1 ELSE 0 END) AS time_control_is_dash
    
FROM games_raw

""").show()

┌──────────────┬───────────────────┬──────────────────────┐
│ base_is_null │ increment_is_null │ time_control_is_dash │
│    int128    │      int128       │        int128        │
├──────────────┼───────────────────┼──────────────────────┤
│        20924 │             20924 │                20924 │
└──────────────┴───────────────────┴──────────────────────┘



`TimeControl` is stored as `base+increment` in seconds
(e.g. `300+3`). There are 856 distinct values. The most common are `60+0` (882,796 games),
`300+0` (661,672), and `180+0` (613,479), followed by a long tail of custom clocks.
Split each value on `+` with `split_part` and converted both parts with `TRY_CAST`.
20,924 games (0.44%) fail to parse, and every one of them is `-` (no clock, which means
correspondence chess). **Decision:** keep these games but label them "Correspondence"
in the speed category (5c), rather than forcing them into a clock-based category.
Correspondence uses different rules (opening books allowed) and its own rating pool,
so it will be analyzed separately from timed games.

### 5c: Speed categories

In [13]:
database_connection.sql("""

WITH parsed_time_controls AS (
    SELECT
        TimeControl,
        Event,
        TRY_CAST(split_part(TimeControl, '+', 1) AS INTEGER) AS base_seconds,
        TRY_CAST(split_part(TimeControl, '+', 2) AS INTEGER) AS increment_seconds,
        TRY_CAST(split_part(TimeControl, '+', 1) AS INTEGER) + 40*(TRY_CAST(split_part(TimeControl, '+', 2) AS INTEGER)) AS estimated_duration_seconds
    FROM games_raw
)
SELECT 
    CASE 
        WHEN base_seconds IS NULL THEN 'Correspondence'
        WHEN estimated_duration_seconds <= 29 THEN 'UltraBullet'
        WHEN estimated_duration_seconds <= 179 THEN 'Bullet'
        WHEN estimated_duration_seconds <= 479 THEN 'Blitz'
        WHEN estimated_duration_seconds <= 1499 THEN 'Rapid'
        WHEN estimated_duration_seconds >= 1500 THEN 'Classical'
        ELSE 'Unknown'
    END AS speed_category,
    COUNT(*) AS games,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS percent_of_games
FROM parsed_time_controls
GROUP BY speed_category
ORDER BY games DESC;


""").show()

┌────────────────┬─────────┬──────────────────┐
│ speed_category │  games  │ percent_of_games │
│    varchar     │  int64  │      double      │
├────────────────┼─────────┼──────────────────┤
│ Blitz          │ 1896675 │            39.76 │
│ Bullet         │ 1622629 │            34.01 │
│ Rapid          │ 1143800 │            23.98 │
│ Classical      │   86329 │             1.81 │
│ Correspondence │   20924 │             0.44 │
└────────────────┴─────────┴──────────────────┘



In [14]:
database_connection.sql("""

WITH parsed_time_controls AS (
    SELECT
        TimeControl,
        Event,
        TRY_CAST(split_part(TimeControl, '+', 1) AS INTEGER) AS base_seconds,
        TRY_CAST(split_part(TimeControl, '+', 2) AS INTEGER) AS increment_seconds,
        TRY_CAST(split_part(TimeControl, '+', 1) AS INTEGER) + 40*(TRY_CAST(split_part(TimeControl, '+', 2) AS INTEGER)) AS estimated_duration_seconds
    FROM games_raw
)
SELECT
    split_part(Event, ' ', 2) AS event_label_2016,
    CASE 
        WHEN base_seconds IS NULL THEN 'Correspondence'
        WHEN estimated_duration_seconds <= 29 THEN 'UltraBullet'
        WHEN estimated_duration_seconds <= 179 THEN 'Bullet'
        WHEN estimated_duration_seconds <= 479 THEN 'Blitz'
        WHEN estimated_duration_seconds <= 1499 THEN 'Rapid'
        WHEN estimated_duration_seconds >= 1500 THEN 'Classical'
        ELSE 'Unknown'
    END AS speed_category,
    COUNT(*) AS games,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS percent_of_games
FROM parsed_time_controls
GROUP BY speed_category, event_label_2016
ORDER BY speed_category, event_label_2016 DESC;


""").show()

┌──────────────────┬────────────────┬─────────┬──────────────────┐
│ event_label_2016 │ speed_category │  games  │ percent_of_games │
│     varchar      │    varchar     │  int64  │      double      │
├──────────────────┼────────────────┼─────────┼──────────────────┤
│ Blitz            │ Blitz          │ 1896675 │            39.76 │
│ Bullet           │ Bullet         │ 1622629 │            34.01 │
│ Classical        │ Classical      │   86329 │             1.81 │
│ Correspondence   │ Correspondence │   20924 │             0.44 │
│ Classical        │ Rapid          │ 1143800 │            23.98 │
└──────────────────┴────────────────┴─────────┴──────────────────┘



As of 10/14/26, Lichess FAQ states that "Lichess time controls are based on estimated game duration = (clock initial time in seconds) + 40 × (clock increment)." They are sorted into categories:
- ≤ 29s = UltraBullet
- ≤ 179s = Bullet
- ≤ 479s = Blitz
- ≤ 1499s = Rapid
- ≥ 1500s = Classical

Since the data I am using is from 2016, none of the event labels are Rapid because this was not its own category in 2016. This results in all of the games that would be considered Rapid in 2026 to be grouped under the event name Classical in 2016. 93% of games labeled “Classical” in 2016 (1,143,800 of 1,230,129) are Rapid by today’s definitions. Additionally, no games in this month qualify as UltraBullet. The games with a - time control, and the 2016 “Correspondence” label match exactly (20,924 each), which confirms the previous decision to keep in separate category.

**Findings**
- Blitz: 1,896,675 games (39.76%)
- Bullet: 1,622,629 games (34.01%)
- Rapid: 1,143,800 games (23.98%)
- Classical: 86,329 games (1.81%)
- Correspondence: 20,924 games (0.44%)
  
**Decision** Use the modern categories, derived from TimeControl, rather than the Event labels. Why: consistent with Lichess’s current definitions, and it separates Rapid from Classical, which the 2016 labels merged.

## Check 6: Date range

## Check 7: Openings